<div style="padding: 28px 32px;
            font-family: 'Georgia', serif;">

  <p style="margin:0; font-size:13px; letter-spacing:3px; text-transform:uppercase; color:#b0477d;">
     Machine Learning · CA1 
  </p>

  <h1 style="margin:8px 0 4px 0; font-size:32px;">
    Gymnastics Prediction CA1
  </h1>

  <p style="margin:4px 0; font-size:15px;"><b>Student:</b> María Fernanda López Díaz</p>
  <p style="margin:4px 0; font-size:15px;"><b>Student ID:</b> 2026794</p>
  <p style="margin:4px 0; font-size:15px;"><b>Lecturer:</b> Mohamed </p>
  <p style="margin:4px 0; font-size:15px;"><b>GitHub:</b>
    <a href="https://github.com/CCT-Dublin/hdipdata-sept26-ft-stage-1-sem1-data-prep-ca1-mafs794" style="color:#b0477d;">
      github.com/CCT-Dublin/hdipdata-sept26-ft-stage-1-sem1-data-prep-ca1-mafs794t</a></p>
</div>

In [14]:
import pandas as pd 
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
raw_data = pd.read_csv("data/data_2022_2023.csv")

<div style="border-left:6px solid #f06fa8; border-radius:12px;
            padding:16px 22px; margin-top:24px; font-family:'Georgia', serif;">
  <h2 style="margin:0 0 6px 0; color:#7a4a6a;">1. Data Preparation</h2>
</div>

In [16]:
raw_data.head()

,LastName,FirstName,Gender,Country,Date,Competition,Round,Location,Apparatus,Rank,D_Score,E_Score,Penalty,Score
0,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",HB,18.0,3.9,8.266,NaN,12.166
1,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",PB,23.0,3.9,6.900,NaN,10.800
2,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",PH,33.0,4.2,6.666,NaN,10.866
3,AAS,Fredrik,m,NOR,23-26 Feb 2023,2023 Cottbus World Cup,qual,"Cottbus, Germany",HB,39.0,4.6,6.700,NaN,11.300
4,AAS,Fredrik,m,NOR,23-26 Feb 2023,2023 Cottbus World Cup,qual,"Cottbus, Germany",PH,44.0,4.4,7.800,NaN,12.200


In [17]:
raw_data.shape

(24434, 14)

The dataset contains 24,434 observations and 14 features. This is the hole dataset, as we are goinf to predict if a gymnast is making in into finals, we are going to base de prediction only with those gymnasts that are in the round qual (short for qualification round or qualifier).

In [8]:
raw_data['Round'].unique()

<StringArray>
['qual', 'AAfinal', 'TeamFinal', 'final', 'TeamQual', 'AAqual']
Length: 6, dtype: str

In [18]:
qual_data = raw_data[raw_data['Round'] == 'qual']

In [19]:
qual_data['Apparatus'].unique()

<StringArray>
['HB', 'PB', 'PH', 'FX', 'SR', 'VT1', 'VT2', 'BB', 'UB', 'hb', 'VT']
Length: 11, dtype: str

VT1 AND VT2 ar abbreviations for VT, we are going to merge it because it means that is the first round and second round for Vault. 'hb' IS HB (Higher Bar) in lower case.

In [21]:
qual_data["Apparatus"] = qual_data["Apparatus"].replace({"VT1": "VT", "VT2": "VT"})

Using only the quad Round we obtain a dataset with 16,058 observations and 14 features. 

In [20]:
qual_data["Apparatus"] = qual_data["Apparatus"].replace({"hb": "HB"})

In [22]:
qual_data.isnull().sum()

LastName           0
FirstName          3
Gender             0
Country            0
Date               0
Competition        0
Round              0
Location           0
Apparatus          0
Rank             699
D_Score            0
E_Score            0
Penalty        12847
Score              0
dtype: int64

We have 3 columns with missing values: "First Name" "Rank" and "Penalty". 
We are going to drop the null values in Rank because without a Rank we cannot make our target variable "top8". Regarding Penalty we have to investigate a little more. 

In [24]:
qual_data['Penalty'].unique()

array([nan, 0.3, 0.1, 0.4, 0.9, 0. , 0.2, 4.3, 0.6, 3. , 4. , 5.3, 0.5,
       8. , 0.7, 3.3, 6.5, 2. , 6. , 1. , 8.1, 0.8, 4.5, 2.1, 6.3, 5. ,
       8.5])

In [32]:
#check if Score column (D_Score + E_Scoreis equal to Score in rows where penalty is null
#this will help us identify if there is a data entry error in the Score column or penalty column
score_check = qual_data[qual_data['Penalty'].isnull() & (qual_data['D_Score'] + qual_data['E_Score'] == qual_data['Score'])]

In [33]:
score_check.shape

(9843, 14)

We obtain only 9,843 out of 12,847 with null in penalty. As we are not able to identify where the problem is (data entry error or what) we are going to drop the columns where the score doesn't match. 

In [34]:
#drop the columns that are have a null value in penalty and are not in the score_check dataframe
qual_data = qual_data.drop(qual_data[qual_data['Penalty'].isnull() & ~qual_data.index.isin(score_check.index)].index)

In [36]:
#drop rows where Rank is null
qual_data = qual_data.drop(qual_data[qual_data['Rank'].isnull()].index)

In [39]:
#change null values in Penalty column to 0
qual_data['Penalty'] = qual_data['Penalty'].fillna(0)

In [40]:
qual_data.isnull().sum()

LastName       0
FirstName      3
Gender         0
Country        0
Date           0
Competition    0
Round          0
Location       0
Apparatus      0
Rank           0
D_Score        0
E_Score        0
Penalty        0
Score          0
dtype: int64

In [41]:
#create the target variable for the model 
#1 if Rank <= 8, else 0
qual_data['Top_8'] = np.where(qual_data['Rank'] <= 8, 1, 0)

In [42]:
qual_data.head(10)

,LastName,FirstName,Gender,Country,Date,Competition,Round,Location,Apparatus,Rank,D_Score,E_Score,Penalty,Score,Top_8
0,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",HB,18.0,3.9,8.266,0.0,12.166,0
1,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",PB,23.0,3.9,6.900,0.0,10.800,0
2,AAS,Fredrik,m,NOR,24-27 Feb 2022,2022 Cottbus World Cup,qual,"Cottbus, Germany",PH,33.0,4.2,6.666,0.0,10.866,0
3,AAS,Fredrik,m,NOR,23-26 Feb 2023,2023 Cottbus World Cup,qual,"Cottbus, Germany",HB,39.0,4.6,6.700,0.0,11.300,0
4,AAS,Fredrik,m,NOR,23-26 Feb 2023,2023 Cottbus World Cup,qual,"Cottbus, Germany",PH,44.0,4.4,7.800,0.0,12.200,0
5,AAS,Fredrik,m,NOR,1-5 Aug 2023,2023 FISU World University Games,qual,"Chengdu, China",FX,54.0,4.0,8.566,0.0,12.566,0
6,AAS,Fredrik,m,NOR,1-5 Aug 2023,2023 FISU World University Games,qual,"Chengdu, China",HB,44.0,4.6,8.166,0.0,12.766,0
8,AAS,Fredrik,m,NOR,1-5 Aug 2023,2023 FISU World University Games,qual,"Chengdu, China",PH,58.0,4.4,7.266,0.0,11.666,0
9,AAS,Fredrik,m,NOR,1-5 Aug 2023,2023 FISU World University Games,qual,"Chengdu, China",SR,73.0,3.5,8.366,0.0,11.866,0
10,AAS,Fredrik,m,NOR,1-5 Aug 2023,2023 FISU World University Games,qual,"Chengdu, China",VT,69.0,4.0,8.966,0.0,12.966,0


In [43]:
#drop Rank and Score columns since we will not use them in the model
qual_data = qual_data.drop(columns=['Rank', 'Score'])

<div style="border-left:6px solid #f06fa8; border-radius:12px;
            padding:16px 22px; margin-top:24px; font-family:'Georgia', serif;">
  <h2 style="margin:0 0 6px 0; color:#7a4a6a;">2. EDA</h2>
</div>